# ESMC Fine-Tuning - Modal Version

Fine-tune ESMC on Modal's serverless H100 GPU infrastructure with persistent model caching.

**What it does:**
- Run fine-tuning remotely on Modal H100 GPU
- Custom LoRA adapters for your classification task
- No local GPU required
- ~4 minutes for 1000 training steps
- Cost-effective: ~$0.13 per fine-tuning run

**Advantages over Colab version:**
- More reliable (no Colab timeout limits)
- Persistent model caching (faster subsequent runs)
- Better for production workflows


In [ ]:
!pip install -q modal pandas numpy tqdm matplotlib

In [ ]:
import modal
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from tqdm import tqdm
from IPython.display import display, Markdown
import time
import pickle

print(f"Modal version: {modal.__version__}")
print("Dependencies loaded successfully")

## Modal Authentication

In [ ]:
#@markdown Get API token from https://modal.com/account/tokens

from getpass import getpass

MODAL_API_TOKEN = getpass("Modal API Token: ")

print("✓ Modal token configured")

## Define Modal App for Fine-Tuning

In [ ]:
# Modal app
app = modal.App("esmc-finetuning")

MINUTES = 60

# Persistent volume for model caching
models_dir = "/root/.cache/huggingface"
models_volume = modal.Volume.from_name(
    "esmc-finetuning-cache",
    create_if_missing=True
)

# Output volume to retrieve fine-tuned adapters
output_dir = "/root/finetuning_output"
output_volume = modal.Volume.from_name(
    "esmc-finetuning-output",
    create_if_missing=True
)

# Container image
image = (
    modal.Image.debian_slim(python_version="3.13")
    .apt_install("git")
    .pip_install(
        "torch>=2.2.0",
        "transformers",
        "peft",
        "accelerate",
        "einops",
        "numpy",
        "pandas",
    )
    .env({
        "HF_HOME": models_dir,
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
    })
    .run_commands(
        f"pip install -q git+https://github.com/Biohub/esm.git@main",
        gpu="H100"
    )
)

print("✓ Modal app configured")
print("✓ Container image defined (H100 GPU)")
print("✓ Volumes created for caching and output")

In [ ]:
@app.cls(
    image=image,
    volumes={
        models_dir: models_volume,
        output_dir: output_volume,
    },
    gpu="H100",
    timeout=45 * MINUTES,
    concurrency_limit=1,
)
class ESMCFinetuner:
    """Remote ESMC fine-tuning on Modal H100 GPU."""
    
    @modal.enter()
    def setup(self):
        """Initialize dependencies."""
        import torch
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    
    @modal.method()
    def finetune(
        self,
        train_sequences: List[str],
        train_labels: List[int],
        val_sequences: Optional[List[str]] = None,
        val_labels: Optional[List[int]] = None,
        num_labels: int = 7,
        num_train_steps: int = 1000,
        learning_rate: float = 1e-4,
        lora_r: int = 8,
        batch_size: int = 8,
    ) -> Dict:
        """Fine-tune ESMC with LoRA on provided data.
        
        Args:
            train_sequences: List of protein sequences
            train_labels: List of class labels
            val_sequences: Optional validation sequences
            val_labels: Optional validation labels
            num_labels: Number of output classes
            num_train_steps: Training steps
            learning_rate: AdamW learning rate
            lora_r: LoRA rank
            batch_size: Training batch size
        
        Returns:
            Dictionary with training history and metrics
        """
        import torch
        import torch.nn.functional as F
        from torch.optim import AdamW
        from torch.cuda.amp import autocast
        from peft import LoraConfig, get_peft_model
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"\nFine-tuning on {device}")
        print(f"Training samples: {len(train_sequences)}")
        if val_sequences:
            print(f"Validation samples: {len(val_sequences)}")
        
        # Load tokenizer and model
        print("Loading model...")
        tokenizer = AutoTokenizer.from_pretrained("biohub/ESMC-300M")
        model = AutoModelForSequenceClassification.from_pretrained(
            "biohub/ESMC-300M",
            num_labels=num_labels,
            device_map="auto"
        )
        
        # Setup LoRA
        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=16,
            lora_dropout=0.01,
            target_modules=["out_proj"],
            task_type="SEQ_CLS",
        )
        model = get_peft_model(model, lora_config)
        print(f"✓ Model loaded with LoRA (trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params)")
        
        # Training loop
        optimizer = AdamW(model.parameters(), lr=learning_rate)
        model.train()
        
        train_history = {"loss": [], "accuracy": []}
        val_history = {"loss": [], "accuracy": []}
        
        print(f"\nTraining {num_train_steps} steps (batch size {batch_size})...")
        
        step = 0
        epoch = 0
        while step < num_train_steps:
            indices = np.random.permutation(len(train_sequences))
            
            for batch_start in range(0, len(indices), batch_size):
                if step >= num_train_steps:
                    break
                
                batch_end = min(batch_start + batch_size, len(indices))
                batch_idx = indices[batch_start:batch_end]
                
                # Prepare batch
                seqs = [train_sequences[i] for i in batch_idx]
                lbls = [train_labels[i] for i in batch_idx]
                
                encodings = tokenizer(
                    seqs,
                    max_length=1024,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt"
                )
                
                batch = {
                    "input_ids": encodings["input_ids"].to(device),
                    "attention_mask": encodings["attention_mask"].to(device),
                    "labels": torch.tensor(lbls, device=device)
                }
                
                # Forward pass with bfloat16
                with autocast(dtype=torch.bfloat16):
                    outputs = model(**batch)
                    loss = outputs.loss
                
                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                # Metrics
                logits = outputs.logits.detach()
                preds = torch.argmax(logits, dim=1)
                accuracy = (preds == batch["labels"]).float().mean().item()
                
                train_history["loss"].append(loss.item())
                train_history["accuracy"].append(accuracy)
                
                if (step + 1) % 250 == 0:
                    print(f"Step {step+1}/{num_train_steps}: loss={loss.item():.3f}, acc={accuracy:.3f}")
                
                # Validation
                if (step + 1) % 250 == 0 and val_sequences:
                    model.eval()
                    with torch.no_grad():
                        encodings = tokenizer(
                            val_sequences,
                            max_length=1024,
                            padding="max_length",
                            truncation=True,
                            return_tensors="pt"
                        )
                        batch = {
                            "input_ids": encodings["input_ids"].to(device),
                            "attention_mask": encodings["attention_mask"].to(device),
                            "labels": torch.tensor(val_labels, device=device)
                        }
                        with autocast(dtype=torch.bfloat16):
                            outputs = model(**batch)
                        val_loss = outputs.loss.item()
                        preds = torch.argmax(outputs.logits, dim=1)
                        val_acc = (preds == batch["labels"]).float().mean().item()
                        val_history["loss"].append(val_loss)
                        val_history["accuracy"].append(val_acc)
                        print(f"  Val: loss={val_loss:.3f}, acc={val_acc:.3f}")
                    model.train()
                
                step += 1
            
            epoch += 1
        
        # Save adapters
        import os
        adapters_path = os.path.join(output_dir, "lora_adapters")
        model.save_pretrained(adapters_path)
        print(f"\n✓ LoRA adapters saved to {adapters_path}")
        
        return {
            "train_history": train_history,
            "val_history": val_history,
            "final_train_loss": float(train_history["loss"][-1]),
            "final_train_acc": float(train_history["accuracy"][-1]),
            "num_steps": step,
        }

print("✓ ESMCFinetuner class defined")

## Configuration & Data Preparation

In [ ]:
# Training configuration
CONFIG = {
    "num_labels": 7,           # EC classification (7 enzyme classes)
    "num_train_steps": 1000,   # ~4 min on H100
    "learning_rate": 1e-4,
    "lora_r": 8,
    "batch_size": 8,
}

OUTPUT_DIR = "/content/finetuning_modal_output"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

EC_CLASSES = [
    "Oxidoreductases",
    "Transferases",
    "Hydrolases",
    "Lyases",
    "Isomerases",
    "Ligases",
    "Translocases"
]

print(f"Config: {CONFIG}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Create sample dataset (or load your own)
def create_sample_dataset(num_samples: int = 500):
    np.random.seed(42)
    
    sample_seqs = [
        "MKVLIVAALLLAVGLAFWECEKRKYQCPEKPQE",
        "MDVFMGVGVVDAKALVDYLVPGQDTAV",
        "MVSGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTG",
        "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
    ]
    
    sequences = []
    labels = []
    
    for i in range(num_samples):
        base_seq = np.random.choice(sample_seqs)
        seq = list(base_seq)
        for _ in range(np.random.randint(0, 3)):
            idx = np.random.randint(0, len(seq))
            seq[idx] = np.random.choice(list("ACDEFGHIKLMNPQRSTVWY"))
        sequences.append("".join(seq))
        labels.append(np.random.randint(0, 7))
    
    return sequences, labels

# Create dataset
train_sequences, train_labels = create_sample_dataset(400)
val_sequences, val_labels = create_sample_dataset(50)

print(f"Train set: {len(train_sequences)} sequences")
print(f"Val set: {len(val_sequences)} sequences")
print(f"Example sequence: {train_sequences[0][:50]}...")

## Run Remote Fine-Tuning

In [ ]:
print("\n🚀 Submitting fine-tuning job to Modal H100...\n")

finetuner = ESMCFinetuner()
result = finetuner.finetune.remote(
    train_sequences=train_sequences,
    train_labels=train_labels,
    val_sequences=val_sequences,
    val_labels=val_labels,
    **CONFIG
)

print(f"\n✓ Fine-tuning completed!")
print(f"  Final train loss: {result['final_train_loss']:.3f}")
print(f"  Final train acc: {result['final_train_acc']:.3f}")
print(f"  Steps: {result['num_steps']}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot loss
ax = axes[0]
ax.plot(result["train_history"]["loss"], alpha=0.5, label="Raw")
if len(result["train_history"]["loss"]) >= 20:
    rolling_loss = pd.Series(result["train_history"]["loss"]).rolling(20).mean()
    ax.plot(rolling_loss, linewidth=2, label="Rolling avg (20)")
if result["val_history"]["loss"]:
    ax.plot(
        np.linspace(0, len(result["train_history"]["loss"]), len(result["val_history"]["loss"])),
        result["val_history"]["loss"],
        'o-', linewidth=2, markersize=6, label="Validation"
    )
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot accuracy
ax = axes[1]
ax.plot(result["train_history"]["accuracy"], alpha=0.5, label="Raw")
if len(result["train_history"]["accuracy"]) >= 20:
    rolling_acc = pd.Series(result["train_history"]["accuracy"]).rolling(20).mean()
    ax.plot(rolling_acc, linewidth=2, label="Rolling avg (20)")
if result["val_history"]["accuracy"]:
    ax.plot(
        np.linspace(0, len(result["train_history"]["accuracy"]), len(result["val_history"]["accuracy"])),
        result["val_history"]["accuracy"],
        'o-', linewidth=2, markersize=6, label="Validation"
    )
ax.set_xlabel("Training Step")
ax.set_ylabel("Accuracy")
ax.set_title("Training & Validation Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_metrics.png", dpi=150, bbox_inches='tight')
plt.show()

# Save results
with open(f"{OUTPUT_DIR}/results.json", "w") as f:
    json.dump(result, f, indent=2, default=str)

print(f"✓ Results saved to {OUTPUT_DIR}/")

## Retrieve Fine-Tuned Adapters

In [ ]:
print("Fine-tuned LoRA adapters are saved on Modal's persistent volume.")
print("\nTo use them later:")
print("""
```python
from peft import PeftModel
from transformers import AutoModelForSequenceClassification

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    "biohub/ESMC-300M",
    num_labels=7
)

# Load LoRA adapters (from Modal output volume or download)
model = PeftModel.from_pretrained(base_model, "./lora_adapters")
model.eval()
```
""")

print(f"\nTo download from Modal:")
print(f"1. Mount the output volume in another Modal job")
print(f"2. Or use `modal volume ls esmc-finetuning-output` to list files")

## Next Steps

In [ ]:
display(Markdown("""
## Using Fine-Tuned Adapters

### **Load from Modal Volume**

The LoRA adapters are saved on Modal's persistent volume (`esmc-finetuning-output`).

```bash
# List files in output volume
modal volume ls esmc-finetuning-output

# Download locally
modal volume get esmc-finetuning-output /root/finetuning_output ./my_adapters
```

### **Use in Python**

```python
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("biohub/ESMC-300M")

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    "biohub/ESMC-300M",
    num_labels=7
)

# Load fine-tuned adapters
model = PeftModel.from_pretrained(base_model, "./lora_adapters")
model.eval()

# Inference
with torch.no_grad():
    encodings = tokenizer(["MKVLIA..."], return_tensors="pt", padding=True)
    outputs = model(**encodings)
    probs = F.softmax(outputs.logits, dim=-1)
    pred_class = torch.argmax(probs).item()
```

### **Training Tips**

- **Increase steps for better convergence**: Change `num_train_steps` to 5000+
- **Larger LoRA rank for more capacity**: Set `lora_r=16` or `32`
- **Custom data**: Load from CSV with columns `sequence`, `label`
- **Regression tasks**: Modify loss function and `num_labels=1`
- **GPU options**: Change from H100 to A100 or A40 for cost savings

### **Scaling to Multiple Fine-Tuning Jobs**

For parallel fine-tuning of different models:
- Increase Modal `concurrency_limit` in `@app.cls()`
- Or submit multiple independent Modal jobs
- Each job gets its own H100 with isolated fine-tuning
"""))